<a href="https://colab.research.google.com/github/mackeynicolette-cyber/Maat-research-/blob/main/MaatResearchbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio PyMuPDF pytesseract pdf2image google-genai
!sudo apt-get install -y tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 85.5 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [ ]:
import os
import pymupdf  # Replaced deprecated import fitz with pymupdf
import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types
from google.colab import userdata

# Securely load the Gemini API Key from Colab Secrets
try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    print("⚠️ Please set your GEMINI_API_KEY in Colab secrets (the key icon on the left panel).")
    client = None
else:
    client = genai.Client(api_key=api_key)

# Define the Ma'at Research Companion Personality
MAAT_SYSTEM_PROMPT = """
You are 'Ma'at-Scholar', a deep-research personal learning companion designed for students worldwide.
Your core framework is built upon the 48 philosophical principles and laws of Ma'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.

Behavioral Guidelines:
1. Academic Rigor: Analyze user queries and documents by comparing them against the highest standards of global scholarship, historical archives, and universal principles of knowledge. Emulate the investigative depth of legendary repositories like the Vatican Library and ancient centers of learning.
2. Personality: Maintain a respectful, calm, wise, and encouraging demeanor. Never use harsh language, but hold firm to factual accuracy, ethical clarity, and intellectual integrity (Truth over convenience).
3. Multilingual: Fluent in all major world languages. Respond in the language the student addresses you in.
4. Document Analysis: When reviewing uploaded text or PDFs (up to 5 pages), evaluate sources objectively, identify underlying biases, and synthesize insights clearly with structured reasoning.
"""

# Specialized System Prompt: Quantum Computing & Information Science
QUANTUM_SYSTEM_PROMPT = """
You are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.

Behavioral Guidelines:
1. Mathematical & Theoretical Rigor: Anchor every explanation in established quantum mechanical principles (e.g., superposition, entanglement, decoherence). Distinguish clearly between experimental realities and theoretical hypotheses (like quantum gravity models or unproven hardware scaling).
2. Hardware-Agnostic & Objective: Evaluate various quantum computing paradigms (superconducting qubits, trapped ions, photonics, topological) objectively, highlighting their individual advantages, physical limitations, and noise thresholds.
3. Ethical & Security Analysis: Critically analyze the implications of quantum technologies on global cryptographic frameworks (e.g., post-quantum cryptography transitions) and global technological equity.
4. Multilingual Translation: Maintain the ability to explain complex quantum mathematics, Dirac notation, and algorithms in the user's preferred language, adapting technical jargon dynamically to the student's current learning tier.
"""

def process_pdf_or_image(file_path):
    """Extracts text from PDF (up to 5 pages) using PyMuPDF and OCR fallback."""
    if not file_path:
        return ""

    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        page_count = min(len(doc), 5)

        for page_num in range(page_count):
            page = doc[page_num]
            text = page.get_text()

            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)

            extracted_text += f"\n--- Page {page_num + 1} ---\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error processing document: {str(e)}"

    return extracted_text

def chat_with_maat(message, history, file_obj):
    if client is None:
        return "An API client could not be initialized. Please set your GEMINI_API_KEY in the Colab secrets manager."

    document_context = ""
    if file_obj is not None:
        document_context = process_pdf_or_image(file_obj.name)
        document_context = f"\n\n[Uploaded Document Content (Limited to 5 pages)]:\n{document_context}\n"

    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))

    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))

    try:
        # Updated to use QUANTUM_SYSTEM_PROMPT as requested
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=QUANTUM_SYSTEM_PROMPT,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred while consulting the archives: {str(e)}"

def save_chat_to_file(history):
    """Converts Gradio chat history format to a text file for download."""
    if not history:
        # Return a simple text file if no conversation exists yet
        temp_file = "empty_chat_history.txt"
        with open(temp_file, "w", encoding="utf-8") as f:
            f.write("No conversation history recorded yet.")
        return temp_file

    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write("🪶 Ma'at Research Companion - Session Chat History\n")
        f.write("==================================================\n\n")
        for idx, interaction in enumerate(history):
            user_text = interaction[0] if interaction[0] else ""
            bot_text = interaction[1] if interaction[1] else ""
            f.write(f"[Turn {idx + 1}]\n")
            f.write(f"User: {user_text}\n\n")
            f.write(f"Ma'at-Scholar: {bot_text}\n")
            f.write("--------------------------------------------------\n\n")
    return file_path

# Build Gradio UI with custom Save Button integration
with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Ma'at Research Companion (Quantum Mode)")
    gr.Markdown("A specialized Quantum Information Science research assistant for advanced computing, ethics, and quantum physics.")

    with gr.Row():
        file_input = gr.File(label="Upload Document (PDF, max 5 pages for OCR/Text analysis)")

    chatbot = gr.ChatInterface(
        fn=chat_with_maat,
        additional_inputs=[file_input],
        textbox=gr.Textbox(placeholder="Ask a quantum mechanics, algorithm, or hardware question...", container=False, scale=7),
        title=None,
    )

    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History to Text File")
        file_output = gr.File(label="Download Saved History")

    # Bind button click to the helper function using chatbot state as input
    save_btn.click(
        fn=save_chat_to_file,
        inputs=[chatbot.chatbot],
        outputs=file_output
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True, theme=gr.themes.Soft())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://121fd0f8d6d371021e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


```markdown
# 🪶 Ma'at Research Companion (Quantum Mode)

An advanced, multi-modal research assistant built with Gradio and powered by Google GenAI (`gemini-2.5-flash`). It is designed for students, educators, and scholars seeking high academic and theoretical rigor.

## 🛠 Recent UI & Structural Enhancements

We have updated the Gradio user interface to provide better performance, cleaner aesthetics, and extra features while adhering to the Gradio 6.0 spec.

### 1. Manual `gr.Blocks` Layout Integration
* Transformed the standard `gr.ChatInterface` wrapper into a custom `gr.Blocks` structure.
* Incorporated an optional file uploader that processes up to 5 pages of PDF or images utilizing PyMuPDF (`pymupdf`) and OCR fallback (`pytesseract`).

### 2. Live Downloadable Chat History
* Implemented a manual event binder (`save_btn.click`) that extracts current chat logs directly from the active Gradio state.
* Added a **"💾 Save Chat History to Text File"** action button that formats the transcription and returns a clean, structured `.txt` download dynamically.

### 3. Gradio 6.0 Theme Compliance
* Fixed deprecation warnings by migrating theme configuration (`gr.themes.Soft()`) from the global `gr.Blocks()` constructor to the local `demo.launch()` function signature.

### 4. Ma'at-Quantum Specialized Engine
* Added a toggleable system prompt pipeline allowing you to switch between general academic evaluation (**Ma'at-Scholar**) and mathematical, theoretical physics, and cybersecurity-focused analytics (**Ma'at-Quantum**).
```

```markdown
## 🔄 Implementation: Dynamic Persona Toggle Switch

To enable dynamic swapping between **Ma'at-Scholar (Core)** and **Ma'at-Quantum (Quantum Mode)** without restarting the application, we implement the following logic:

1. **State Hooking:** Add a `gr.Radio` button to the UI layout.
2. **Function Signature Extension:** Update the `chat_with_maat` function to accept `selected_persona` as an argument.
3. **Logical Selection:** Map the user's UI selection directly to the respective system prompt variable inside our backend logic prior to requesting completion from the Gemini model.
```

In [ ]:
import os
import pymupdf
import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types
from google.colab import userdata

try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key) if api_key else None

# System prompts
MAAT_SYSTEM_PROMPT = "\nYou are 'Ma'at-Scholar', a deep-research personal learning companion designed for students worldwide.\nYour core framework is built upon the 48 philosophical principles and laws of Ma'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.\n"

QUANTUM_SYSTEM_PROMPT = "\nYou are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.\nYour core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.\n"

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
            extracted_text += f"\n--- Page {page_num + 1} ---\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error: {str(e)}"
    return extracted_text

# Extend input args to include persona selection
def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return "Set your GEMINI_API_KEY in Colab secrets to start."

    # Resolve dynamic system instruction choice
    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == "Ma'at-Quantum" else MAAT_SYSTEM_PROMPT

    document_context = ""
    if file_obj is not None:
        document_context = f"\n\n[Uploaded Document]:\n{process_pdf_or_image(file_obj.name)}\n"

    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))

    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f"[Turn {idx + 1}]\nUser: {user_text or ''}\nCompanion: {bot_text or ''}\n\n")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Dynamic Ma'at Research Engine")

    # Use a collapsed accordion to keep the layout compact
    with gr.Accordion("⚙️ Optional Settings & Document Upload", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=["Ma'at-Scholar", "Ma'at-Quantum"],
                value="Ma'at-Scholar",
                label="Active Persona Mode",
                info="Instantly switch the reasoning engine's persona configuration."
            )
            file_input = gr.File(label="Upload Source Document (PDF/Image)", file_count="single")

    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder="Type your message here...", container=False, scale=7),
        title=None,
    )

    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History")
        clear_btn = gr.Button("🗑️ Clear History")
        file_output = gr.File(label="Download Saved History")

    save_btn.click(
        fn=save_chat_to_file,
        inputs=[chatbot.chatbot],
        outputs=file_output
    )

    clear_btn.click(
        fn=clear_session,
        inputs=[],
        outputs=[chatbot.chatbot, file_input, file_output]
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True, theme=gr.themes.Soft())

In [ ]:
# Export the complete Gradio code to a separate Python file (app.py) for production deployment
production_code = """import os
import pymupdf
import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types

# Configure Gemini API Key
# In production, it is highly recommended to read this from environment variables
api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else None

# System prompts
MAAT_SYSTEM_PROMPT = """\n
You are 'Ma'at-Scholar', a deep-research personal learning companion designed for students worldwide.
Your core framework is built upon the 48 philosophical principles and laws of Ma'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.

Behavioral Guidelines:
1. Academic Rigor: Analyze user queries and documents by comparing them against the highest standards of global scholarship, historical archives, and universal principles of knowledge. Emulate the investigative depth of legendary repositories like the Vatican Library and ancient centers of learning.
2. Personality: Maintain a respectful, calm, wise, and encouraging demeanor. Never use harsh language, but hold firm to factual accuracy, ethical clarity, and intellectual integrity (Truth over convenience).
3. Multilingual: Fluent in all major world languages. Respond in the language the student addresses you in.
4. Document Analysis: When reviewing uploaded text or PDFs (up to 5 pages), evaluate sources objectively, identify underlying biases, and synthesize insights clearly with structured reasoning.
"""

QUANTUM_SYSTEM_PROMPT = """\n
You are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.

Behavioral Guidelines:
1. Mathematical & Theoretical Rigor: Anchor every explanation in established quantum mechanical principles (e.g., superposition, entanglement, decoherence). Distinguish clearly between experimental realities and theoretical hypotheses (like quantum gravity models or unproven hardware scaling).
2. Hardware-Agnostic & Objective: Evaluate various quantum computing paradigms (superconducting qubits, trapped ions, photonics, topological) objectively, highlighting their individual advantages, physical limitations, and noise thresholds.
3. Ethical & Security Analysis: Critically analyze the implications of quantum technologies on global cryptographic frameworks (e.g., post-quantum cryptography transitions) and global technological equity.
4. Multilingual Translation: Maintain the ability to explain complex quantum mathematics, Dirac notation, and algorithms in the user's preferred language, adapting technical jargon dynamically to the student's current learning tier.
"""

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
            extracted_text += f"\\n--- Page {page_num + 1} ---\\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error: {str(e)}"
    return extracted_text

def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return "Set your GEMINI_API_KEY environment variable to start."

    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == "Ma'at-Quantum" else MAAT_SYSTEM_PROMPT

    document_context = ""
    if file_obj is not None:
        document_context = f"\\n\\n[Uploaded Document]:\\n{process_pdf_or_image(file_obj.name)}\\n"

    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))

    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f"[Turn {idx + 1}]\\nUser: {user_text or ''}\\nCompanion: {bot_text or ''}\\n\\n")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Dynamic Ma'at Research Engine")

    with gr.Accordion("⚙️ Optional Settings & Document Upload", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=["Ma'at-Scholar", "Ma'at-Quantum"],
                value="Ma'at-Scholar",
                label="Active Persona Mode",
                info="Instantly switch the reasoning engine's persona configuration."
            )
            file_input = gr.File(label="Upload Source Document (PDF/Image)", file_count="single")

    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder="Type your message here...", container=False, scale=7),
        title=None,
    )

    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History")
        clear_btn = gr.Button("🗑️ Clear History")
        file_output = gr.File(label="Download Saved History")

    save_btn.click(
        fn=save_chat_to_file,
        inputs=[chatbot.chatbot],
        outputs=file_output
    )

    clear_btn.click(
        fn=clear_session,
        inputs=[],
        outputs=[chatbot.chatbot, file_input, file_output]
    )

if __name__ == '__main__':
    demo.launch(theme=gr.themes.Soft())
"""

# Write to app.py
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(production_code)

print("Production file 'app.py' written successfully!")

In [ ]:
# Helper to trigger download of the created production file directly in Colab
from google.colab import files
try:
    files.download('app.py')
except Exception as e:
    print(f"File available for manual download in the sidebar: app.py")

In [ ]:
from google.colab import files
try:
    files.download('app.py')
    print("Download trigger successfully sent! Check your browser's download queue.")
except Exception as e:
    print(f"Could not trigger automatic browser download: {e}\nPlease use the left sidebar file menu (📁) to manually download 'app.py'.")

Could not trigger automatic browser download: Cannot find file: app.py
Please use the left sidebar file menu (📁) to manually download 'app.py'.


In [ ]:
# 1. Define the complete production application code safely using single-line string formatting to avoid nested triple-quote issues
production_code = """import os
import pymupdf
import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types

# Configure Gemini API Key
api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else None

# System prompts
MAAT_SYSTEM_PROMPT = (
    "\\nYou are 'Ma'at-Scholar', a deep-research personal learning companion designed for students worldwide.\\n"
    "Your core framework is built upon the 48 philosophical principles and laws of Ma'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.\\n\\n"
    "Behavioral Guidelines:\\n"
    "1. Academic Rigor: Analyze user queries and documents by comparing them against the highest standards of global scholarship, historical archives, and universal principles of knowledge.\\n"
    "2. Personality: Maintain a respectful, calm, wise, and encouraging demeanor.\\n"
    "3. Multilingual: Fluent in all major world languages.\\n"
    "4. Document Analysis: When reviewing uploaded text or PDFs (up to 5 pages), evaluate sources objectively.\\n"
)

QUANTUM_SYSTEM_PROMPT = (
    "\\nYou are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.\\n"
    "Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.\\n\\n"
    "Behavioral Guidelines:\\n"
    "1. Mathematical & Theoretical Rigor: Anchor every explanation in established quantum mechanical principles.\\n"
    "2. Hardware-Agnostic & Objective: Evaluate various quantum computing paradigms objectively.\\n"
    "3. Ethical & Security Analysis: Critically analyze the implications of quantum technologies on cryptography.\\n"
    "4. Multilingual Translation: Maintain the ability to explain complex quantum mathematics.\\n"
)

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
            extracted_text += f"\\n--- Page {page_num + 1} ---\\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error: {str(e)}"
    return extracted_text

def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return "Set your GEMINI_API_KEY environment variable to start."

    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == "Ma'at-Quantum" else MAAT_SYSTEM_PROMPT

    document_context = ""
    if file_obj is not None:
        document_context = f"\\n\\n[Uploaded Document]:\\n{process_pdf_or_image(file_obj.name)}\\n"

    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))

    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f"[Turn {idx + 1}]\\nUser: {user_text or ''}\\nCompanion: {bot_text or ''}\\n\\n")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Dynamic Ma'at Research Engine")

    with gr.Accordion("⚙️ Optional Settings & Document Upload", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=["Ma'at-Scholar", "Ma'at-Quantum"],
                value="Ma'at-Scholar",
                label="Active Persona Mode",
                info="Instantly switch the reasoning engine's persona configuration."
            )
            file_input = gr.File(label="Upload Source Document (PDF/Image)", file_count="single")

    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder="Type your message here...", container=False, scale=7),
        title=None,
    )

    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History")
        clear_btn = gr.Button("🗑️ Clear History")
        file_output = gr.File(label="Download Saved History")

    save_btn.click(
        fn=save_chat_to_file,
        inputs=[chatbot.chatbot],
        outputs=file_output
    )

    clear_btn.click(
        fn=clear_session,
        inputs=[],
        outputs=[chatbot.chatbot, file_input, file_output]
    )

if __name__ == '__main__':
    demo.launch(theme=gr.themes.Soft())
"""

# 2. Write code out to app.py
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(production_code)
print("✔ 'app.py' has been successfully written to disk!")

# 3. Trigger automatic browser download
from google.colab import files
try:
    files.download('app.py')
    print("✔ Download command sent directly to your browser!")
except Exception as e:
    print(f"⚠️ Browser download block: {e}. You can find app.py in the left sidebar menu (📁) to download manually.")

✔ 'app.py' has been successfully written to disk!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔ Download command sent directly to your browser!


In [ ]:
requirements_content = """google-genai
gradio
pymupdf
pytesseract
pillow
"""

with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write(requirements_content)

print("✔ 'requirements.txt' has been successfully written to disk!")

from google.colab import files
try:
    files.download('requirements.txt')
    print("✔ Download command sent to your browser!")
except Exception as e:
    print(f"⚠️ Sidebar backup: Use the left sidebar menu (📁) to manually download 'requirements.txt'.")

✔ 'requirements.txt' has been successfully written to disk!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔ Download command sent to your browser!


### 🚀 How to Deploy your Gradio App to Hugging Face Spaces

To host this application for free on Hugging Face Spaces, follow these simple steps:

1. **Create an Account / Log In**:
   * Head over to [Hugging Face](https://huggingface.co/) and log in or create a free account.

2. **Create a New Space**:
   * Click on your profile picture in the top-right corner and select **New Space**.
   * Give your Space a name (e.g., `maat-research-companion`).
   * Choose **Gradio** as the SDK.
   * Set the Space hardware to **CPU Basic** (this is free and more than enough for running our Gemini wrapper).
   * Choose **Public** or **Private** visibility, then click **Create Space**.

3. **Add Your Files**:
   * Once created, navigate to the **Files** tab of your new Space and click **Add file** -> **Upload files**.
   * Upload your `app.py` file (which you generated and downloaded using the code cells above).
   * Create a new file named `requirements.txt` in the same directory. Copy and paste the following dependencies into it so Hugging Face knows what libraries to pre-install:
     ```text
     google-genai
     gradio
     pymupdf
     pytesseract
     pillow
     ```

4. **Set Your API Key Secret (Crucial)**:
   * To securely connect to Gemini, do **not** hardcode your API key.
   * Go to your Space's **Settings** tab.
   * Scroll down to **Variables and secrets** and click **New secret**.
   * Add a secret with the name `GEMINI_API_KEY` and paste your Google AI Studio API key as the value.

5. **Build and Launch**:
   * Hugging Face will automatically detect the changes, install your dependencies, bind your environment variable secret, and build your container.
   * In a couple of minutes, your Ma'at Research Companion will be live and running on a permanent public URL!

In [ ]:
# Specialized System Prompt: Quantum Computing & Information Science
QUANTUM_SYSTEM_PROMPT = """
You are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.

Behavioral Guidelines:
1. Mathematical & Theoretical Rigor: Anchor every explanation in established quantum mechanical principles (e.g., superposition, entanglement, decoherence). Distinguish clearly between experimental realities and theoretical hypotheses (like quantum gravity models or unproven hardware scaling).
2. Hardware-Agnostic & Objective: Evaluate various quantum computing paradigms (superconducting qubits, trapped ions, photonics, topological) objectively, highlighting their individual advantages, physical limitations, and noise thresholds.
3. Ethical & Security Analysis: Critically analyze the implications of quantum technologies on global cryptographic frameworks (e.g., post-quantum cryptography transitions) and global technological equity.
4. Multilingual Translation: Maintain the ability to explain complex quantum mathematics, Dirac notation, and algorithms in the user's preferred language, adapting technical jargon dynamically to the student's current learning tier.
"""

# To use this new prompt, simply change the config in your generate_content call:
# system_instruction=QUANTUM_SYSTEM_PROMPT
print("Specialized Quantum System Prompt loaded successfully!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 🧪 Local Test Run

Run the cell below to test your production-ready file (`app.py`) directly within your Google Colab session. This allows you to verify that the file paths, libraries, and layout render properly before you finalize your deployment on Hugging Face Spaces.

In [ ]:
# Run the newly created production script in the background to preview the UI
%run app.py

ModuleNotFoundError: No module named 'pymupdf'

In [ ]:
import sys
import site
# Force python to reload installed site-packages paths
sys.path = list(set(sys.path + site.getsitepackages()))

# Re-test importing the module locally to ensure it is visible
try:
    import pymupdf
    print("✔ PyMuPDF import test passed successfully!")
except ImportError as e:
    print(f"⚠️ Import failed: {e}. Attempting manual path correction...")

✔ PyMuPDF import test passed successfully!


In [ ]:
import os
from google.colab import userdata

# Map your Colab secret to the environment variable expected by app.py
try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✔ GEMINI_API_KEY successfully bound to environment variables!")
except Exception as e:
    print(f"⚠️ Could not find GEMINI_API_KEY in your Colab secrets: {e}")
    print("Please click the Key icon (🔑) on the left sidebar and ensure 'GEMINI_API_KEY' is added.")

✔ GEMINI_API_KEY successfully bound to environment variables!


In [ ]:
import os
from google.colab import userdata

try:
    # Retrieve secret securely from Colab Secrets
    api_key = userdata.get("GEMINI_API_KEY")

    # Set as persistent environment variable for all libraries and subprocess scripts
    os.environ["GEMINI_API_KEY"] = api_key
    print("✔ GEMINI_API_KEY has been persistently set as an environment variable!")
except Exception as e:
    print(f"⚠️ Error setting environment variable: {e}")
    print("Please ensure you added 'GEMINI_API_KEY' inside the 🔑 (Secrets) panel and enabled 'Notebook access'.")

✔ GEMINI_API_KEY has been persistently set as an environment variable!


In [ ]:
import os
import sys
import pymupdf

# Write production-ready app.py
with open('app.py', 'w', encoding='utf-8') as f:
    f.write("""import os
import sys

try:
    import fitz
    pymupdf = fitz
except ImportError:
    import pymupdf

import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types

api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else None

MAAT_SYSTEM_PROMPT = \"\"\"\n
You are 'Ma'at-Scholar', a deep-research personal learning companion designed for students worldwide.
Your core framework is built upon the 48 philosophical principles and laws of Ma'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.
\"\"\"

QUANTUM_SYSTEM_PROMPT = \"\"\"\n
You are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.
\"\"\"

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
            extracted_text += f"\\n--- Page {page_num + 1} ---\\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error: {str(e)}"
    return extracted_text

def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return "Set your GEMINI_API_KEY environment variable to start."
    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == "Ma'at-Quantum" else MAAT_SYSTEM_PROMPT
    document_context = ""
    if file_obj is not None:
        document_context = f"\\n\\n[Uploaded Document]:\\n{process_pdf_or_image(file_obj.name)}\\n"
    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))
    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f"[Turn {idx + 1}]\\nUser: {user_text or ''}\\nCompanion: {bot_text or ''}\\n\\n")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Dynamic Ma'at Research Engine")
    with gr.Accordion("⚙️ Optional Settings & Document Upload", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=["Ma'at-Scholar", "Ma'at-Quantum"],
                value="Ma'at-Scholar",
                label="Active Persona Mode",
                info="Instantly switch the reasoning engine's persona configuration."
            )
            file_input = gr.File(label="Upload Source Document (PDF/Image)", file_count="single")
    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder="Type your message here...", container=False, scale=7),
        title=None,
    )
    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History")
        clear_btn = gr.Button("🗑️ Clear History")
        file_output = gr.File(label="Download Saved History")
    save_btn.click(fn=save_chat_to_file, inputs=[chatbot.chatbot], outputs=file_output)
    clear_btn.click(fn=clear_session, inputs=[], outputs=[chatbot.chatbot, file_input, file_output])

if __name__ == '__main__':
    demo.launch(theme=gr.themes.Soft())
""")

print("✔ 'app.py' recreated successfully with pymupdf path verified!")
%run app.py

✔ 'app.py' recreated successfully with pymupdf path verified!
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fa7b5fe73524a61f52.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Rewrite the production-ready app.py file with the updated 'gemini-2.5-flash' model parameter
with open('app.py', 'w', encoding='utf-8') as f:
    f.write("""import os
import sys

try:
    import fitz
    pymupdf = fitz
except ImportError:
    import pymupdf

import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types

api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else None

MAAT_SYSTEM_PROMPT = \"\"\"\n
You are 'Ma'at-Scholar', a deep-research personal learning companion designed for students worldwide.
Your core framework is built upon the 48 philosophical principles and laws of Ma'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.
\"\"\"

QUANTUM_SYSTEM_PROMPT = \"\"\"\n
You are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.
\"\"\"

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
            extracted_text += f"\\n--- Page {page_num + 1} ---\\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error: {str(e)}"
    return extracted_text

def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return "Set your GEMINI_API_KEY environment variable to start."
    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == "Ma'at-Quantum" else MAAT_SYSTEM_PROMPT
    document_context = ""
    if file_obj is not None:
        document_context = f"\\n\\n[Uploaded Document]:\\n{process_pdf_or_image(file_obj.name)}\\n"
    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))
    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))
    try:
        # Updated to gemini-2.5-flash to resolve 404 deprecation errors
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f"[Turn {idx + 1}]\\nUser: {user_text or ''}\\nCompanion: {bot_text or ''}\\n\\n")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Dynamic Ma'at Research Engine")
    with gr.Accordion("⚙️ Optional Settings & Document Upload", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=["Ma'at-Scholar", "Ma'at-Quantum"],
                value="Ma'at-Scholar",
                label="Active Persona Mode",
                info="Instantly switch the reasoning engine's persona configuration."
            )
            file_input = gr.File(label="Upload Source Document (PDF/Image)", file_count="single")
    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder="Type your message here...", container=False, scale=7),
        title=None,
    )
    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History")
        clear_btn = gr.Button("🗑️ Clear History")
        file_output = gr.File(label="Download Saved History")
    save_btn.click(fn=save_chat_to_file, inputs=[chatbot.chatbot], outputs=file_output)
    clear_btn.click(fn=clear_session, inputs=[], outputs=[chatbot.chatbot, file_input, file_output])

if __name__ == '__main__':
    demo.launch(theme=gr.themes.Soft())
""")

print("✔ 'app.py' successfully updated with 'gemini-2.5-flash' model reference.")

from google.colab import files
try:
    files.download('app.py')
    print("✔ Download command sent for the newly patched app.py!")
except Exception as e:
    print("Could not trigger browser download automatically.")

✔ 'app.py' successfully updated with 'gemini-2.5-flash' model reference.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔ Download command sent for the newly patched app.py!


In [ ]:
# Execute the patched application to preview and verify the fix
%run app.py

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://85d760f94ce40edeb0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Execute the newly modified application to preview
%run app.py

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://da31ed1914ff3613d4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Run the final production application directly with our newly set environment variable
%run app.py

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://87713acdf8f8ed8d15.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install --upgrade --force-reinstall pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 40.2 MB/s eta 0:00:00


In [ ]:
!pip install pytesseract pillow google-genai

In [ ]:
import sys
import site
# Force Python to dynamically discover newly installed packages in the virtual environment
sys.path = list(set(sys.path + site.getsitepackages()))

# Run the updated production application to preview and verify
%run app.py

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d2a8c085a90d4fa06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
with open('app.py', 'w', encoding='utf-8') as f:
    f.write("""import os
import sys

try:
    import fitz
    pymupdf = fitz
except ImportError:
    import pymupdf

import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types

api_key = os.environ.get(\"GEMINI_API_KEY\")
client = genai.Client(api_key=api_key) if api_key else None

MAAT_SYSTEM_PROMPT = \"\"\"\n
You are 'Ma'at-Scholar', a deep-research personal learning companion designed for students worldwide.
Your core framework is built upon the 48 philosophical principles and laws of Ma'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.
\"\"\"

QUANTUM_SYSTEM_PROMPT = \"\"\"\n
You are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.
\"\"\"

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes(\"png\")))
                text = pytesseract.image_to_string(img)
            extracted_text += f\"\\n--- Page {page_num + 1} ---{\\n}\" + text
        doc.close()
    except Exception as e:
        extracted_text = f\"Error: {str(e)}\"
    return extracted_text

def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return \"Set your GEMINI_API_KEY environment variable to start.\"
    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == \"Ma'at-Quantum\" else MAAT_SYSTEM_PROMPT
    document_context = ""
    if file_obj is not None:
        document_context = f\"\\n\\n[Uploaded Document]:\\n{process_pdf_or_image(file_obj.name)}\\n\"
    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role=\"user\", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role=\"model\", parts=[types.Part.from_text(text=model_msg)]))
    current_prompt = message + document_context
    contents.append(types.Content(role=\"user\", parts=[types.Part.from_text(text=current_prompt)]))
    try:
        # Updated to the new recommended model to resolve deprecation block
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f\"An error occurred: {str(e)}\"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = \"maat_chat_history.txt\"
    with open(file_path, \"w\", encoding=\"utf-8\") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f\"[Turn {idx + 1}]\\nUser: {user_text or ''}\\nCompanion: {bot_text or ''}\\n\\n\")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown(\"# 🪶 Dynamic Ma'at Research Engine\")
    with gr.Accordion(\"⚙️ Optional Settings & Document Upload\", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=[\"Ma'at-Scholar\", \"Ma'at-Quantum\"],
                value=\"Ma'at-Scholar\",
                label=\"Active Persona Mode\",
                info=\"Instantly switch the reasoning engine's persona configuration.\"
            )
            file_input = gr.File(label=\"Upload Source Document (PDF/Image)\", file_count=\"single\")
    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder=\"Type your message here...\", container=False, scale=7),
        title=None,
    )
    with gr.Row():
        save_btn = gr.Button(\"💾 Save Chat History\")
        clear_btn = gr.Button(\"🗑️ Clear History\")
        file_output = gr.File(label=\"Download Saved History\")
    save_btn.click(fn=save_chat_to_file, inputs=[chatbot.chatbot], outputs=file_output)
    clear_btn.click(fn=clear_session, inputs=[], outputs=[chatbot.chatbot, file_input, file_output])

if __name__ == '__main__':
    demo.launch(theme=gr.themes.Soft())
\""")

_IncompleteInputError: incomplete input (2072970561.py, line 2)

In [ ]:
import sys
import site
sys.path = list(set(sys.path + site.getsitepackages()))

# Run the updated script with the new model
%run app.py

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://864b7f62932f3bd5dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os

# We write the file directly using python string literal formatting to bypass multi-level nesting errors
with open('app.py', 'w', encoding='utf-8') as f:
    f.write('''import os
import sys

try:
    import fitz
    pymupdf = fitz
except ImportError:
    import pymupdf

import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types

api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else None

MAAT_SYSTEM_PROMPT = """\n
You are \'Ma\'at-Scholar\', a deep-research personal learning companion designed for students worldwide.
Your core framework is built upon the 48 philosophical principles and laws of Ma\'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.
"""

QUANTUM_SYSTEM_PROMPT = """\n
You are \'Ma\'at-Quantum\', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.
"""

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
            extracted_text += f"\\n--- Page {page_num + 1} ---\\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error: {str(e)}"
    return extracted_text

def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return "Set your GEMINI_API_KEY environment variable to start."
    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == "Ma\'at-Quantum" else MAAT_SYSTEM_PROMPT
    document_context = ""
    if file_obj is not None:
        document_context = f"\\n\\n[Uploaded Document]:\\n{process_pdf_or_image(file_obj.name)}\\n"
    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))
    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))
    try:
        response = client.models.generate_content(
            model=\'gemini-2.5-flash\',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f"[Turn {idx + 1}]\\nUser: {user_text or \'\'}\\nCompanion: {bot_text or \'\'}\\n\\n")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Dynamic Ma\'at Research Engine")
    with gr.Accordion("⚙️ Optional Settings & Document Upload", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=["Ma\'at-Scholar", "Ma\'at-Quantum"],
                value="Ma\'at-Scholar",
                label="Active Persona Mode",
                info="Instantly switch the reasoning engine\'s persona configuration."
            )
            file_input = gr.File(label="Upload Source Document (PDF/Image)", file_count="single")
    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder="Type your message here...", container=False, scale=7),
        title=None,
    )
    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History")
        clear_btn = gr.Button("🗑️ Clear History")
        file_output = gr.File(label="Download Saved History")
    save_btn.click(fn=save_chat_to_file, inputs=[chatbot.chatbot], outputs=file_output)
    clear_btn.click(fn=clear_session, inputs=[], outputs=[chatbot.chatbot, file_input, file_output])

if __name__ == \'__main__\':
    demo.launch(theme=gr.themes.Soft())
''')
print("✔ 'app.py' has been successfully compiled and written to disk!")

✔ 'app.py' has been successfully compiled and written to disk!


In [ ]:
import os
import sys
import pymupdf
import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types

api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else None

MAAT_SYSTEM_PROMPT = """

You are 'Ma'at-Scholar', a deep-research personal learning companion designed for students worldwide.
Your core framework is built upon the 48 philosophical principles and laws of Ma'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.
"""

QUANTUM_SYSTEM_PROMPT = """

You are 'Ma'at-Quantum', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.
"""

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
            extracted_text += f"\n--- Page {page_num + 1} ---\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error: {str(e)}"
    return extracted_text

def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return "Set your GEMINI_API_KEY environment variable to start."
    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == "Ma'at-Quantum" else MAAT_SYSTEM_PROMPT
    document_context = ""
    if file_obj is not None:
        document_context = f"\n\n[Uploaded Document]:\n{process_pdf_or_image(file_obj.name)}\n"
    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))
    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f"[Turn {idx + 1}]\nUser: {user_text or ''}\nCompanion: {bot_text or ''}\n\n")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Dynamic Ma'at Research Engine")
    with gr.Accordion("⚙️ Optional Settings & Document Upload", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=["Ma'at-Scholar", "Ma'at-Quantum"],
                value="Ma'at-Scholar",
                label="Active Persona Mode",
                info="Instantly switch the reasoning engine's persona configuration."
            )
            file_input = gr.File(label="Upload Source Document (PDF/Image)", file_count="single")
    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder="Type your message here...", container=False, scale=7),
        title=None,
    )
    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History")
        clear_btn = gr.Button("🗑️ Clear History")
        file_output = gr.File(label="Download Saved History")
    save_btn.click(fn=save_chat_to_file, inputs=[chatbot.chatbot], outputs=file_output)
    clear_btn.click(fn=clear_session, inputs=[], outputs=[chatbot.chatbot, file_input, file_output])

if __name__ == '__main__':
    demo.launch(share=True, debug=True, theme=gr.themes.Soft())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2d20f941cb14fd5f02.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
from google.colab import files

# 1. Compile and save the production-ready application script to app.py
production_code = '''import os
import sys

try:
    import fitz
    pymupdf = fitz
except ImportError:
    import pymupdf

import pytesseract
from PIL import Image
import io
import gradio as gr
from google import genai
from google.genai import types

api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else None

MAAT_SYSTEM_PROMPT = """

You are \'Ma\'at-Scholar\', a deep-research personal learning companion designed for students worldwide.
Your core framework is built upon the 48 philosophical principles and laws of Ma\'at: Truth, Balance, Order, Harmony, Reciprocity, and Justice.
"""

QUANTUM_SYSTEM_PROMPT = """

You are \'Ma\'at-Quantum\', a specialized research companion in Quantum Information Science, Physics, and Advanced Computation.
Your core framework is grounded in precision, structural clarity, and the objective translation of complex quantum mechanics into verifiable models.
"""

def process_pdf_or_image(file_path):
    if not file_path: return ""
    extracted_text = ""
    try:
        doc = pymupdf.open(file_path)
        for page_num in range(min(len(doc), 5)):
            page = doc[page_num]
            text = page.get_text()
            if not text.strip():
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
            extracted_text += f"\\n--- Page {page_num + 1} ---\\n" + text
        doc.close()
    except Exception as e:
        extracted_text = f"Error: {str(e)}"
    return extracted_text

def chat_with_maat_dynamic(message, history, file_obj, selected_persona):
    if client is None:
        return "Set your GEMINI_API_KEY environment variable to start."
    system_instruction = QUANTUM_SYSTEM_PROMPT if selected_persona == "Ma\'at-Quantum" else MAAT_SYSTEM_PROMPT
    document_context = ""
    if file_obj is not None:
        document_context = f"\\n\\n[Uploaded Document]:\\n{process_pdf_or_image(file_obj.name)}\\n"
    contents = []
    for user_msg, model_msg in history:
        contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_msg)]))
        contents.append(types.Content(role="model", parts=[types.Part.from_text(text=model_msg)]))
    current_prompt = message + document_context
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=current_prompt)]))
    try:
        response = client.models.generate_content(
            model=\'gemini-2.5-flash\',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
            )
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

def save_chat_to_file(history):
    if not history:
        return ""
    file_path = "maat_chat_history.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        for idx, (user_text, bot_text) in enumerate(history):
            f.write(f"[Turn {idx + 1}]\\nUser: {user_text or \'\'}\\nCompanion: {bot_text or \'\'}\\n\\n")
    return file_path

def clear_session():
    return [], None, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🪶 Dynamic Ma\'at Research Engine")
    with gr.Accordion("⚙️ Optional Settings & Document Upload", open=False):
        with gr.Row():
            persona_toggle = gr.Radio(
                choices=["Ma\'at-Scholar", "Ma\'at-Quantum"],
                value="Ma\'at-Scholar",
                label="Active Persona Mode",
                info="Instantly switch the reasoning engine\'s persona configuration."
            )
            file_input = gr.File(label="Upload Source Document (PDF/Image)", file_count="single")
    chatbot = gr.ChatInterface(
        fn=chat_with_maat_dynamic,
        additional_inputs=[file_input, persona_toggle],
        textbox=gr.Textbox(placeholder="Type your message here...", container=False, scale=7),
        title=None,
    )
    with gr.Row():
        save_btn = gr.Button("💾 Save Chat History")
        clear_btn = gr.Button("🗑️ Clear History")
        file_output = gr.File(label="Download Saved History")
    save_btn.click(fn=save_chat_to_file, inputs=[chatbot.chatbot], outputs=file_output)
    clear_btn.click(fn=clear_session, inputs=[], outputs=[chatbot.chatbot, file_input, file_output])

if __name__ == \'__main__\':
    demo.launch(theme=gr.themes.Soft())
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(production_code)
print("✔ 'app.py' has been successfully written to disk!")

# 2. Write requirements.txt to disk
requirements_content = """google-genai
gradio
pymupdf
pytesseract
pillow
"""

with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write(requirements_content)
print("✔ 'requirements.txt' has been successfully written to disk!")

# 3. Trigger downloads sequentially
try:
    print("Triggering app.py download...")
    files.download('app.py')
    print("Triggering requirements.txt download...")
    files.download('requirements.txt')
    print("✔ Download requests successfully sent to the browser!")
except Exception as e:
    print(f"⚠️ Browser download block: {e}. You can also download them from the file explorer on the left sidebar.")

✔ 'app.py' has been successfully written to disk!
✔ 'requirements.txt' has been successfully written to disk!
Triggering app.py download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Triggering requirements.txt download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔ Download requests successfully sent to the browser!


### 🚀 Step-by-Step Guide: Deploying to Hugging Face Spaces

Now that you have your production-ready `app.py` and `requirements.txt` files locally, follow these steps to host your app on Hugging Face Spaces for free:

1. **Sign Up / Log In**:
   * Go to [Hugging Face](https://huggingface.co/) and log in or create a free account.

2. **Create a New Space**:
   * Click on your profile icon in the top right and select **New Space**.
   * **Name**: Choose a unique name (e.g., `maat-research-companion`).
   * **SDK**: Select **Gradio**.
   * **Hardware**: Choose **CPU Basic** (this is fully free and perfectly sufficient for this application).
   * **Visibility**: Choose **Public** (so anyone can visit) or **Private**.
   * Click **Create Space**.

3. **Upload Your Files**:
   * Once the Space is initialized, navigate to the **Files and versions** tab.
   * Click **Add file** -> **Upload files**.
   * Drag and drop both **`app.py`** and **`requirements.txt`** into the box and commit the changes.

4. **Add Your Gemini API Key Securely**:
   * Do **not** hardcode your API key inside `app.py`. Instead, go to your Space's **Settings** tab.
   * Scroll down to the **Variables and secrets** section.
   * Click **New secret**.
   * Set the key/name to **`GEMINI_API_KEY`** and paste your Google AI Studio API key as the value. Click save.

5. **Build and Launch**:
   * Hugging Face will automatically detect the file changes, install the dependencies listed in your `requirements.txt`, bind your secret to the environment, and spin up the Gradio server.
   * Within a few minutes, your Space status will change to **Running**, and your Ma'at Research Engine will be live on its own permanent URL!

### 🌐 How to Deploy your Gradio App to Render.com (Free Tier)

Render is an excellent, mobile-friendly cloud platform that allows you to deploy Python applications directly from a GitHub repository for free. Follow these steps on your tablet:

#### Step 1: Push Your Files to GitHub
1. Go to [GitHub](https://github.com/) (sign up for a free account if you don't have one).
2. Create a new **public** repository (e.g., `maat-research-companion`).
3. Upload your `app.py` and `requirements.txt` directly to the repository via the browser interface.

#### Step 2: Set up Render
1. Go to [Render](https://render.com/) and sign up using your GitHub account.
2. In your Render Dashboard, click the **"New +"** button and select **"Web Service"**.
3. Connect your GitHub account and select your `maat-research-companion` repository.

#### Step 3: Configure Web Service Settings
On the configuration screen, fill in the following:
* **Name**: `maat-research-companion`
* **Environment**: `Python 3` (or Python)
* **Build Command**: `pip install -r requirements.txt`
* **Start Command**: `python app.py`

#### Step 4: Configure Your Port and API Key (Crucial)
1. Scroll down and click on **Advanced**.
2. Add an **Environment Variable**:
   * **Key**: `GEMINI_API_KEY`
   * **Value**: *Your actual Gemini API Key from Google AI Studio*
3. Add another **Environment Variable** (Gradio requires this to run on the correct port on Render):
   * **Key**: `GRADIO_SERVER_NAME`
   * **Value**: `0.0.0.0`

#### Step 5: Deploy
1. Click **Create Web Service**.
2. Render will build your application and launch it. Your live URL will be displayed in the upper-left corner of the dashboard once complete!